In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re
from constants import KENPOM_FILE
from selenium import webdriver

In [3]:
def gen_browser():
    options = webdriver.FirefoxOptions()
    options.headless = True
    return webdriver.Firefox(options=options)

In [4]:
%%time
url = 'http://kenpom.com/index.php'

# Can also find archived version
# url = "https://web.archive.org/web/20220317101252/https://kenpom.com/"

# This doesn't work anymore
# f = requests.get(url)
# soup = BeautifulSoup(f.text, "lxml")

browser = gen_browser()
browser.get(url)
soup = BeautifulSoup(browser.page_source, "lxml")

CPU times: user 723 ms, sys: 35.1 ms, total: 758 ms
Wall time: 6.2 s


In [5]:
COLUMNS = [
    'Rank', 'Team', 'Conference', 'W-L', 'AdjustEM',
    'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
    'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
    'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
    'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank'
]

In [6]:
df = pd.DataFrame(
    [
        [tr.text for tr in row.find_all('td')]
        for row in soup.find('table', {'id': 'ratings-table'}).find_all(class_=r"tourney")
    ],
    columns=COLUMNS
)

In [7]:
df.shape

(64, 21)

In [8]:
# Returns true if given string is a number and a valid seed number (1-16)
def is_valid_seed(x):
    return str(x).replace(' ', '').isdigit() and int(x) > 0 and int(x) <= 16


def parse_name(row):
    if is_valid_seed(row['Team'][-2:]):
        row['Seed'] = row['Team'][-2:].strip()
        row['Team'] = row['Team'][:-2].strip()
    else:
        row['Seed'] = np.NaN
        row['Team'] = row['Team']
    return row


# Parse out seed/team
df = df.apply(parse_name, axis=1)

In [9]:
# Split W-L column into wins and losses
df['Wins'] = df['W-L'].apply(lambda x: int(re.sub('-.*', '', x)))
df['Losses'] = df['W-L'].apply(lambda x: int(re.sub('.*-', '', x)))
df.drop('W-L', inplace=True, axis=1)

# Reorder columns just cause I'm OCD
df = df[['Rank', 'Seed', 'Team', 'Conference', 'Wins', 'Losses', 'AdjustEM',
         'AdjustO', 'AdjustO Rank', 'AdjustD', 'AdjustD Rank',
         'AdjustT', 'AdjustT Rank', 'Luck', 'Luck Rank',
         'SOS Pyth', 'SOS Pyth Rank', 'SOS OppO', 'SOS OppO Rank',
         'SOS OppD', 'SOS OppD Rank', 'NCSOS Pyth', 'NCSOS Pyth Rank']]

# Drop non tournament teams
df = df.dropna()

In [10]:
df

,Rank,Seed,Team,Conference,Wins,Losses,AdjustEM,AdjustO,AdjustO Rank,AdjustD,...,Luck,Luck Rank,SOS Pyth,SOS Pyth Rank,SOS OppO,SOS OppO Rank,SOS OppD,SOS OppD Rank,NCSOS Pyth,NCSOS Pyth Rank
0,1,1,Duke,ACC,31,3,+38.25,128.0,3,89.8,...,-.019,232,+10.11,56,112.5,47,102.4,63,+9.34,21
1,2,1,Florida,SEC,30,4,+36.19,128.6,1,92.5,...,+.007,158,+14.47,16,114.6,17,100.1,19,-1.98,232
2,3,1,Houston,B12,30,4,+35.41,123.2,10,87.8,...,+.015,139,+13.66,24,113.6,33,99.9,17,+2.57,101
3,4,1,Auburn,SEC,28,5,+35.11,128.5,2,93.4,...,+.030,100,+19.05,2,117.1,2,98.1,2,+10.38,19
4,5,2,Tennessee,SEC,27,7,+31.13,120.3,18,89.1,...,+.016,136,+15.95,7,116.7,4,100.7,34,-0.88,202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,161,14,Montana,BSky,25,9,+0.24,110.7,98,110.5,...,+.186,1,-2.36,205,106.4,180,108.8,239,+8.56,28
60,180,16,Norfolk St.,MEAC,24,10,-1.50,107.2,166,108.7,...,+.056,58,-7.18,325,103.1,306,110.3,336,+3.09,90
61,216,16,SIUE,OVC,22,11,-4.25,102.0,255,106.2,...,+.052,67,-9.01,357,101.4,349,110.4,345,-3.37,276
62,238,16,Mount St. Mary's,MAAC,23,12,-6.03,101.1,279,107.1,...,+.147,2,-7.16,323,102.0,333,109.1,267,-1.16,211


In [11]:
df.to_csv(KENPOM_FILE.format(gender="mens"), index=False)

# Old Version
This doesn't work with NIT seeds

In [ ]:
table_html = soup.find_all('table', {'id': 'ratings-table'})

In [3]:
# Weird issue w/ <thead> in the html
# Prevents us from just using pd.read_html
# Let's find all the thead contents and just replace/remove them
# This allows us to easily put the table row data into a dataframe using pandas
thead = table_html[0].find_all('thead')

table = table_html[0]
for x in thead:
    table = str(table).replace(str(x), '')

In [4]:
#    table = "<table id='ratings-table'>%s</table>" % table
df = pd.read_html(table, converters={3: lambda x: str(x)})[0]

df.columns = COLUMNS